# Laboratorio 7: Pipeline ETL, Profiling y Gobernanza en el Data Lake
**Materia:** Fundamentos de Big Data (IDSD413) - EPN

**Objetivo:** Simular un escenario real de ingesta de datos ruidosos (anomalías, nulos, duplicados), ejecutar un perfilamiento estadístico para dimensionar el caos, y procesarlos a través de un flujo ETL construyendo las capas lógicas de un Data Lake (Capa *Raw* y Capa *Curated*).



In [5]:
import pandas as pd
import numpy as np
import os

print("1/3 Generando datos base perfectos en memoria...")
np.random.seed(42)
N_ROWS = 5_000_000
df_base = pd.DataFrame({
    'id_transaccion': np.arange(N_ROWS),
    'id_cliente': np.random.randint(1, 100000, N_ROWS),
    'monto': np.random.uniform(5.0, 2500.0, N_ROWS),
    'categoria': np.random.choice(['Electronica', 'Ropa', 'Hogar', 'Alimentos', 'Juguetes'], N_ROWS),
    'estado': np.random.choice(['Completado', 'Pendiente', 'Cancelado', 'Reembolsado'], N_ROWS)
})

# Guardamos una base fresca para evitar el error de bytes corruptos en Colab
nombre_fresco = 'base_fresca.parquet'
df_base.to_parquet(nombre_fresco, engine='pyarrow')

print("2/3 Leyendo archivo desde disco e inyectando caos (Simulación de ingesta real)...")
df_original = pd.read_parquet(nombre_fresco, engine='pyarrow')

# a) Inyectamos un 5% de nulos
n_nulos = int(len(df_original) * 0.05)
indices_nulos = np.random.choice(df_original.index, n_nulos, replace=False)
df_original.loc[indices_nulos, 'monto'] = np.nan

# b) Duplicamos 50,000 registros
df_duplicados = df_original.sample(n=50000)
df_ingesta = pd.concat([df_original, df_duplicados], ignore_index=True)

# c) Inyectamos outliers (valores absurdos que rompen la estadística)
df_ingesta.loc[0:100, 'monto'] = 9999999.99

print(f"3/3 ¡Éxito! Dataset de ingesta listo. Total de filas: {len(df_ingesta):,}")

1/3 Generando datos base perfectos en memoria...
2/3 Leyendo archivo desde disco e inyectando caos (Simulación de ingesta real)...
3/3 ¡Éxito! Dataset de ingesta listo. Total de filas: 5,050,000


## Fase 1: Profiling y Capa RAW (El Diagnóstico del Caos)
Antes de transformar, debemos perfilar el archivo crudo para entender la magnitud del problema y luego guardarlo en la capa inmutable del Data Lake.

In [6]:
print("--- DIAGNÓSTICO ESTADÍSTICO (PROFILING) ---\n")

# 1. Auditoría de Nulos y Duplicados
print("1. Valores Nulos por Columna:")
print(df_ingesta.isnull().sum())
print(f"\n2. Registros Duplicados Exactos: {df_ingesta.duplicated().sum():,}")

# 2. Auditoría de Outliers (Fíjense en la diferencia entre el promedio y el max)
print("\n3. Resumen Estadístico de 'monto':")
print(df_ingesta['monto'].describe().apply(lambda x: f"{x:,.2f}"))

# 3. Guardado en Capa RAW (Inmutable)
os.makedirs('./datalake/raw', exist_ok=True)
os.makedirs('./datalake/curated', exist_ok=True)
# RETO 1: Creación de la capa de cuarentena
os.makedirs('./datalake/quarantine', exist_ok=True)

print("\n--- EXTRACT & LOAD (EL) ---")
print("Guardando evidencia en la Capa RAW...")
df_ingesta.to_parquet('./datalake/raw/ingesta_transacciones_sucio.parquet')
print("¡Archivo RAW almacenado con éxito! Listos para limpiar.")

--- DIAGNÓSTICO ESTADÍSTICO (PROFILING) ---

1. Valores Nulos por Columna:
id_transaccion         0
id_cliente             0
monto             252510
categoria              0
estado                 0
dtype: int64

2. Registros Duplicados Exactos: 49,999

3. Resumen Estadístico de 'monto':
count    4,797,490.00
mean         1,463.00
std         45,882.62
min              5.00
25%            628.91
50%          1,252.56
75%          1,876.37
max      9,999,999.99
Name: monto, dtype: object

--- EXTRACT & LOAD (EL) ---
Guardando evidencia en la Capa RAW...
¡Archivo RAW almacenado con éxito! Listos para limpiar.


## Fase 2: Transform (T) - Pipeline ETL de Limpieza y Normalización
Levantamos el dato desde RAW y ejecutamos limpieza masiva: eliminar duplicados, tratar nulos mediante imputación robusta (mediana) y eliminar valores atípicos (outliers) utilizando el Rango Intercuartílico (IQR). Finalmente, normalizamos para Machine Learning.

In [7]:
print("Iniciando Transformación...\n")

# 1. Eliminación de duplicados
antes = len(df_ingesta)
df_limpio = df_ingesta.drop_duplicates().copy()
print(f"Duplicados eliminados: {antes - len(df_limpio):,}")

# 2. RETO 2: Descarte inteligente + imputación condicionada
# 2.a) Descartar filas con estado 'Cancelado' Y monto nulo
mask_descarte = (df_limpio['estado'] == 'Cancelado') & (df_limpio['monto'].isnull())
n_descartadas = mask_descarte.sum()
df_limpio = df_limpio[~mask_descarte].copy()
print(f"Filas 'Cancelado' con monto nulo descartadas: {n_descartadas:,}")

# 2.b) Imputar con mediana por categoría los nulos restantes
#      (Completado, Pendiente, Reembolsado)
medianas = df_limpio.groupby('categoria')['monto'].transform('median')
n_imputados = df_limpio['monto'].isnull().sum()
df_limpio['monto'] = df_limpio['monto'].fillna(medianas)
print(f"Nulos imputados con mediana por categoría: {n_imputados:,}")
print(f"Nulos restantes en monto: {df_limpio['monto'].isnull().sum()}")

# 3. RETO 1: Detección IQR + cuarentena (en vez de borrar)
Q1 = df_limpio['monto'].quantile(0.25)
Q3 = df_limpio['monto'].quantile(0.75)
IQR = Q3 - Q1
limite_superior = Q3 + 1.5 * IQR

mask_outlier = df_limpio['monto'] > limite_superior
df_cuarentena = df_limpio[mask_outlier].copy()
df_cuarentena['motivo_cuarentena'] = f'monto > limite_superior IQR ({limite_superior:,.2f})'

df_cuarentena.to_parquet('./datalake/quarantine/outliers_monto.parquet')
print(f"Outliers enviados a QUARANTINE: {len(df_cuarentena):,}")

df_limpio = df_limpio[~mask_outlier].copy()

# 4. Normalización Min-Max (solo sobre datos ya limpios)
min_monto = df_limpio['monto'].min()
max_monto = df_limpio['monto'].max()
df_limpio['monto_escalado'] = (df_limpio['monto'] - min_monto) / (max_monto - min_monto)
print("Normalización Min-Max completada.")

Iniciando Transformación...

Duplicados eliminados: 49,999
Filas 'Cancelado' con monto nulo descartadas: 62,513
Nulos imputados con mediana por categoría: 187,479
Nulos restantes en monto: 0
Outliers enviados a QUARANTINE: 101
Normalización Min-Max completada.


## Fase 3: Load (Curated Layer) y Verificación Final
Guardamos el dato limpio en formato columnar en la capa *Curated*. Finalmente, comparamos la estadística del dato curado frente al desastre original.

In [8]:
# 1. Guardamos en la capa analítica
df_limpio.to_parquet('./datalake/curated/transacciones_model_ready.parquet')
print("Datos escritos en la Capa CURATED exitosamente.\n")

# 2. Demostración del dato limpio (El "Después")
print("--- ESTADÍSTICA DEL DATO CURADO (Monto limpio vs Escalado) ---")
# Comparamos la estadística final para demostrar que la media volvió a la normalidad
print(df_limpio[['monto', 'monto_escalado']].describe().applymap(lambda x: f"{x:,.2f}"))

print("\n--- MUESTRA DE DATOS LISTOS PARA MACHINE LEARNING ---")
display(df_limpio[['id_transaccion', 'categoria', 'monto', 'monto_escalado']].head(5))

print("\n--- RESUMEN DE ALMACENAMIENTO DEL DATA LAKE ---")
print(f"RAW (Sucio): {os.path.getsize('./datalake/raw/ingesta_transacciones_sucio.parquet') / (1024*1024):.2f} MB")
print(f"CURATED (Limpio): {os.path.getsize('./datalake/curated/transacciones_model_ready.parquet') / (1024*1024):.2f} MB")

Datos escritos en la Capa CURATED exitosamente.

--- ESTADÍSTICA DEL DATO CURADO (Monto limpio vs Escalado) ---
              monto monto_escalado
count  4,937,387.00   4,937,387.00
mean       1,252.50           0.50
std          706.46           0.28
min            5.00           0.00
25%          653.39           0.26
50%        1,252.53           0.50
75%        1,851.88           0.74
max        2,500.00           1.00

--- MUESTRA DE DATOS LISTOS PARA MACHINE LEARNING ---


/tmp/ipykernel_3713/779839698.py:8: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  print(df_limpio[['monto', 'monto_escalado']].describe().applymap(lambda x: f"{x:,.2f}"))


,id_transaccion,categoria,monto,monto_escalado
101,101,Juguetes,1015.580233,0.405042
102,102,Alimentos,1837.507104,0.734472
103,103,Electronica,997.149883,0.397655
104,104,Ropa,2021.055379,0.808038
105,105,Juguetes,1251.907721,0.499762



--- RESUMEN DE ALMACENAMIENTO DEL DATA LAKE ---
RAW (Sucio): 74.41 MB
CURATED (Limpio): 133.05 MB
